In [1]:
import numpy as np

In [2]:
class MLP_Network:

    def __init__(self, layers):

        self.layers = layers
        self.inputs = np.array(layers[0])
        self.outputs = np.array(layers[-1])

        self.weights = np.empty((1, len(layers) - 1), dtype=object)
        for i in range(1, len(layers)):
            temp_array = np.random.randn(self.layers[i], self.layers[i-1])
            self.weights[0, i-1] = temp_array

        self.biases = np.empty((1, len(layers) - 1), dtype=object)
        for i in range(1, len(layers)):
            self.biases[0, i-1] = np.random.randn(self.layers[i], 1)

        self.output_arrays = np.empty((len(layers), 1), dtype=object)

    def sigmoid(self, X):
        return 1/(1+np.exp(-X))

    def forward(self, X):
        if X.shape[0] != self.layers[0] and X.shape[1] == self.layers[0]:
            X = X.T
        elif X.shape[0] == self.layers[0] and X.shape[1] == 1:
            pass
        else:
            print("wrong input array dimensions")
            return

        self.output_arrays[0, 0] = X.copy()
        for step in range(len(self.layers)-1):
            X = self.output_arrays[step, 0]
            W = self.weights[0, step]
            B = self.biases[0, step]
            Z = np.dot(W, X) + B
            A = self.sigmoid(Z)
            self.output_arrays[step+1, 0] = A.copy()
        return self.output_arrays[-1, 0]

    def backward(self, y_true, y_pred, learning_rate):
        if y_pred.shape != y_true.shape:
            y_pred = y_pred.T

        previous_error = (y_pred - y_true)
        for i in range(len(self.layers)-2, -1, -1):
            error = np.dot(previous_error, self.weights[0, i]) * self.output_arrays[i, 0] * (1 - self.output_arrays[i, 0])

            self.weights[0, i] -= learning_rate * np.dot(self.output_arrays[i, 0].T, error)
            self.biases[0, i] -= learning_rate * error
            previous_error = error

    def train(self, X, y_true, learning_rate, epochs):
        for epoch in range(epochs):
            y_pred = self.forward(X)
            self.backward(y_true, y_pred, learning_rate)
            if (epoch+1) % 100 == 0:
                loss = -np.sum(y_true * np.log(y_pred)) / X.shape[0]
                print(f'Epoch {epoch+1}, Loss: {loss:.4f}')

    def print(self):
        print({
            "layers": self.layers,
            "weights": [self.weights[0, i].shape for i in range(self.weights.shape[1])],
            "biases": [self.biases[0, i].shape for i in range(self.biases.shape[1])],
        })

In [3]:
myMLP = MLP_Network([5, 3, 4, 2])
input = np.array(np.random.rand(5)).reshape((1, 5))

In [4]:
output = myMLP.forward(input)
output.shape

(2, 1)

In [5]:
myMLP.backward(np.array([[1, 0]]), output, 0.5)

ValueError: operands could not be broadcast together with shapes (2,1) (4,4) (2,1) 

In [ ]:
with open("MNIST Dataset/archive/train-labels.idx1-ubyte", "rb") as f:
    data = f.read()

# Skip the 8-byte IDX header
labels = np.frombuffer(data, dtype=np.uint8, offset=8)

print(labels.shape)
print(labels[:20])